# Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [2]:
import os
import sys

project_root = os.path.abspath('..')
sys.path.append(project_root)

# Customer Table Analyis

## Conclusions:
The customer dataset has 99,441 rows with 5 attributes and no nulls. The dataset is fine grain with each row having a unique ``customer_id`` which according to the explanation given with the dataset is uniquely generated for each order. ``customer_unique_id`` is not distinct as it is unique for each customer and is thus repeated across orders. 41.98% of orders are from Sao Paolo state with 15% orders in Sao Paolo city.

*Note: Given this datasets grain is based on orders and not customer profiles, its name is currently misleading. To acutaully represent customer data (where the table actually represents the location profiles of unique customers) it would have to be modified to make the ``customer_unique_id`` the primary key and then remove the ``customer_id`` attribute. The utulity of this however depends on the presence of ```customer_unique_id``` in other datasets.*

In [3]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
customers.reset_index()
customers = customers.replace({None: np.nan})
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [5]:
customers.describe(include = "str")

,customer_id,customer_unique_id,customer_city,customer_state
count,99441,99441,99441,99441
unique,99441,96096,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,sao paulo,SP
freq,1,17,15540,41746


In [6]:
customers.duplicated().sum()

np.int64(0)

In [7]:
from src.utils import value_counts_per

value_counts_per(customers.drop_duplicates(subset = "customer_unique_id"), "customer_city")[:5]

,count,percentage
customer_city,,
sao paulo,14971,15.58
rio de janeiro,6611,6.88
belo horizonte,2671,2.78
brasilia,2066,2.15
curitiba,1462,1.52


In [18]:
value_counts_per(customers.drop_duplicates(subset = "customer_unique_id"), "customer_state")[:5]

,count,percentage
customer_state,,
SP,40295,41.93
RJ,12377,12.88
MG,11255,11.71
RS,5277,5.49
PR,4882,5.08


# Geolocation Data

## Conclusions: 

The geolocation dataset has over a million entries, 5 attributes, and no nulls. 

There are more unique cities in the geolocation dataset compared to the customers dataset.

There are multiple duplicated values and multiple co-ordinates per zipcode. So I have applied agreggation where the geographic centroid is calcuted for each unique ```geolocation_zip_code_prefix```.

In [9]:
geo = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
geo .reset_index()
geo  = geo .replace({None: np.nan})
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [ ]:
#No nulls, 5 attributes
geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [ ]:
#geolocation_city unique > customer_city unique
#gelocation_state unique > customer_city unique
geo.describe(include = 'str')

,geolocation_city,geolocation_state
count,1000163,1000163
unique,8011,27
top,sao paulo,SP
freq,135800,404268


In [12]:
geo.nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [ ]:
#Multiple duplicated locations
geo.duplicated().sum()

np.int64(261831)

In [ ]:
#Multiple duplicated co-ordinates?
geo[["geolocation_lat", "geolocation_lng"]].duplicated().sum()

np.int64(281700)

In [15]:
geo.groupby('geolocation_zip_code_prefix')[['geolocation_city', 'geolocation_state']] \
.nunique().sort_values(by='geolocation_city', ascending=False).head(10)

,geolocation_city,geolocation_state
geolocation_zip_code_prefix,,
17970,5,1
13455,5,1
78290,5,1
6900,5,1
13318,5,1
13457,5,1
13454,5,1
28950,5,1
42850,5,1


In [ ]:
#There are up to 746 different latitude and longitutde co-ordinates under each zip code prefix.
geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']] \
.nunique().sort_values(by='geolocation_lat', ascending=False).head(10)

,geolocation_lat,geolocation_lng
geolocation_zip_code_prefix,,
38400,746,745
11680,726,726
35500,726,726
11740,666,666
36400,627,627
39400,618,620
35162,611,611
38408,600,599
37200,595,594


### Calculating a Geographic Centroid for each ```geolocation_zip_code_prefix```

In [ ]:
geo[['geolocation_lat', 'geolocation_lng']] = geo.groupby()